# BEHRT Next Visit Prediction - QUICK VERSION (30 minutes)

**This notebook is optimized for fast fine-tuning:**
- Smaller model matching MLM_QUICK (hidden_size: 144)
- Fewer epochs (10 vs 50)
- Smaller batch size (128 vs 256)
- Uses subset of data (30%)

**Prerequisites:**
- Run MLM_QUICK.ipynb first to get pretrained model

**For production/research, use NextXVisit.ipynb (full version)**

In [1]:
import sys 
sys.path.insert(0, '../')

In [2]:
from torch.utils.data import DataLoader
import pandas as pd
from common.common import create_folder,H5Recorder
import numpy as np
from torch.utils.data.dataset import Dataset
import os
import torch
import torch.nn as nn
import pytorch_pretrained_bert as Bert

from model import optimiser
import sklearn.metrics as skm
import math
from torch.utils.data.dataset import Dataset
import random
import numpy as np
import torch
import time
from sklearn.metrics import roc_auc_score
from common.common import load_obj
from model.utils import age_vocab
from dataLoader.NextXVisit import NextVisit
from model.NextXVisit import BertForMultiLabelPrediction
import warnings
warnings.filterwarnings(action='ignore')

In [3]:
# ============ QUICK VERSION CONFIG ============
file_config = {
    'vocab': '../data/processed/vocab_ccsr',  # ← CHANGE THIS
    'train': '../data/processed/train_nextvisit_ccsr.parquet',  # ← CHANGE THIS
    'test': '../data/processed/test_nextvisit_ccsr.parquet',  # ← CHANGE THIS
}

optim_config = {
    'lr': 3e-5,  # Slightly higher for faster convergence
    'warmup_proportion': 0.1,
    'weight_decay': 0.01
}

global_params = {
    'batch_size': 128,
    'gradient_accumulation_steps': 1,
    'device': 'cpu',
    'output_dir': '../data/models/quick/',
    'best_name': 'behrt_nextvisit_ccsr_quick.pt',
    'max_len_seq': 64,  # ← CHANGE THIS from 48 to 64
    'max_age': 110,
    'age_year': False,
    'age_symbol': None,
    'min_visit': 5,
    'data_fraction': 0.8,
}

# IMPORTANT: Use the quick pretrained model
pretrain_model_path = '../data/models/quick/behrt_mlm_ccsr_quick.pt'

In [4]:
BertVocab = load_obj(file_config['vocab'])
ageVocab, _ = age_vocab(max_age=global_params['max_age'], symbol=global_params['age_symbol'])

In [5]:
def format_label_vocab(token2idx):
    token2idx = token2idx.copy()
    del token2idx['PAD']
    del token2idx['SEP']
    del token2idx['CLS']
    del token2idx['MASK']
    token = list(token2idx.keys())
    labelVocab = {}
    for i,x in enumerate(token):
        labelVocab[x] = i
    return labelVocab

labelVocab = format_label_vocab(BertVocab['token2idx'])

In [6]:
# ============ SMALLER MODEL CONFIGURATION (MATCHES MLM_QUICK) ============
model_config = {
    'vocab_size': len(BertVocab['token2idx'].keys()),  # This will be 478 for CCSR
    'hidden_size': 144,  # ← MUST MATCH your MLM model
    'seg_vocab_size': 2,
    'age_vocab_size': len(ageVocab.keys()),
    'max_position_embedding': global_params['max_len_seq'],
    'hidden_dropout_prob': 0.1,
    'num_hidden_layers': 3,  # ← MUST MATCH your MLM model
    'num_attention_heads': 6,  # ← MUST MATCH your MLM model
    'attention_probs_dropout_prob': 0.1,
    'intermediate_size': 256,  # ← MUST MATCH your MLM model
    'hidden_act': 'gelu',
    'initializer_range': 0.02,
}

feature_dict = {
    'word':True,
    'seg':True,
    'age':True,
    'position': True
}

print("\n=" * 60)
print("QUICK TRAINING CONFIGURATION")
print("=" * 60)
print(f"Model size: ~30% of full model")
print(f"Hidden size: {model_config['hidden_size']} (vs 288 full)")
print(f"Layers: {model_config['num_hidden_layers']} (vs 6 full)")
print(f"Batch size: {global_params['batch_size']} (vs 256 full)")
print(f"Data fraction: {global_params['data_fraction']*100}%")
print(f"Expected training time: 20-30 minutes")
print("=" * 60)


=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
QUICK TRAINING CONFIGURATION
Model size: ~30% of full model
Hidden size: 144 (vs 288 full)
Layers: 3 (vs 6 full)
Batch size: 128 (vs 256 full)
Data fraction: 80.0%
Expected training time: 20-30 minutes


In [7]:
class BertConfig(Bert.modeling.BertConfig):
    def __init__(self, config):
        super(BertConfig, self).__init__(
            vocab_size_or_config_json_file=config.get('vocab_size'),
            hidden_size=config['hidden_size'],
            num_hidden_layers=config.get('num_hidden_layers'),
            num_attention_heads=config.get('num_attention_heads'),
            intermediate_size=config.get('intermediate_size'),
            hidden_act=config.get('hidden_act'),
            hidden_dropout_prob=config.get('hidden_dropout_prob'),
            attention_probs_dropout_prob=config.get('attention_probs_dropout_prob'),
            max_position_embeddings = config.get('max_position_embedding'),
            initializer_range=config.get('initializer_range'),
        )
        self.seg_vocab_size = config.get('seg_vocab_size')
        self.age_vocab_size = config.get('age_vocab_size')

In [8]:
# Load and subsample training data
train = pd.read_parquet(file_config['train'])
print(f"Original training data: {len(train)} samples")
train = train.sample(frac=global_params['data_fraction'], random_state=42)
train = train.reset_index(drop=True) 
print(f"Quick training data: {len(train)} samples ({global_params['data_fraction']*100}%)")

Dset = NextVisit(token2idx=BertVocab['token2idx'], label2idx=labelVocab, age2idx=ageVocab, 
                 dataframe=train, max_len=global_params['max_len_seq'])
trainload = DataLoader(dataset=Dset, batch_size=global_params['batch_size'], shuffle=True, num_workers=3)

Original training data: 143702 samples
Quick training data: 114962 samples (80.0%)


In [9]:
# Load and subsample test data
test = pd.read_parquet(file_config['test'])
print(f"Original test data: {len(test)} samples")
test = test.sample(frac=global_params['data_fraction'], random_state=42)
test = test.reset_index(drop=True) 
print(f"Quick test data: {len(test)} samples ({global_params['data_fraction']*100}%)")

Dset = NextVisit(token2idx=BertVocab['token2idx'], label2idx=labelVocab, age2idx=ageVocab, 
                 dataframe=test, max_len=global_params['max_len_seq'])
testload = DataLoader(dataset=Dset, batch_size=global_params['batch_size'], shuffle=False, num_workers=3)

Original test data: 18244 samples
Quick test data: 14595 samples (80.0%)


In [10]:
conf = BertConfig(model_config)
model = BertForMultiLabelPrediction(conf, num_labels=len(labelVocab.keys()), feature_dict=feature_dict)

In [11]:
def load_model(path, model):
    pretrained_dict = torch.load(path, map_location='cpu')
    model_dict = model.state_dict()
    pretrained_dict = {k: v for k, v in pretrained_dict.items() if k in model_dict}
    model_dict.update(pretrained_dict)
    model.load_state_dict(model_dict)
    return model

print(f"Loading pretrained model from: {pretrain_model_path}")
model = load_model(pretrain_model_path, model)
print("Pretrained model loaded successfully!")

Loading pretrained model from: ../data/models/quick/behrt_mlm_ccsr_quick.pt
Pretrained model loaded successfully!


In [12]:
model = model.to(global_params['device'])
optim = optimiser.adam(params=list(model.named_parameters()), config=optim_config)

t_total value of -1 results in schedule not being applied


In [13]:
import sklearn
def precision(logits, label):
    sig = nn.Sigmoid()
    output=sig(logits)
    label, output=label.cpu(), output.detach().cpu()
    tempprc= sklearn.metrics.average_precision_score(label.numpy(),output.numpy(), average='samples')
    return tempprc, output, label

def precision_test(logits, label):
    sig = nn.Sigmoid()
    output=sig(logits)
    tempprc= sklearn.metrics.average_precision_score(label.numpy(),output.numpy(), average='samples')
    roc = sklearn.metrics.roc_auc_score(label.numpy(),output.numpy(), average='samples')
    return tempprc, roc, output, label

In [14]:
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer(classes=list(labelVocab.values()))
mlb.fit([[each] for each in list(labelVocab.values())])

MultiLabelBinarizer(classes=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,
                             15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27,
                             28, 29, ...])

In [15]:
def train(e):
    model.train()
    tr_loss = 0
    temp_loss = 0
    nb_tr_examples, nb_tr_steps = 0, 0
    cnt = 0
    start = time.time()
    
    for step, batch in enumerate(trainload):
        cnt +=1
        age_ids, input_ids, posi_ids, segment_ids, attMask, targets, _ = batch
        
        targets = torch.tensor(mlb.transform(targets.numpy()), dtype=torch.float32)

        age_ids = age_ids.to(global_params['device'])
        input_ids = input_ids.to(global_params['device'])
        posi_ids = posi_ids.to(global_params['device'])
        segment_ids = segment_ids.to(global_params['device'])
        attMask = attMask.to(global_params['device'])
        targets = targets.to(global_params['device'])
        
        loss, logits = model(input_ids, age_ids, segment_ids, posi_ids,attention_mask=attMask, labels=targets)
        
        if global_params['gradient_accumulation_steps'] >1:
            loss = loss/global_params['gradient_accumulation_steps']
        loss.backward()
        
        temp_loss += loss.item()
        tr_loss += loss.item()
        nb_tr_examples += input_ids.size(0)
        nb_tr_steps += 1
        
        # Print every 50 steps
        if step % 50 == 0:
            prec, a, b = precision(logits, targets)
            elapsed = time.time() - start
            print("epoch: {}\t| Step: {}/{}\t| Loss: {:.4f}\t| APS: {:.4f}\t| Time: {:.1f}s".format(
                e, step, len(trainload), temp_loss/(step+1), prec, elapsed))
        
        if (step + 1) % global_params['gradient_accumulation_steps'] == 0:
            optim.step()
            optim.zero_grad()
    
    epoch_time = time.time() - start
    return epoch_time

def evaluation():
    model.eval()
    y = []
    y_label = []
    tr_loss = 0
    
    for step, batch in enumerate(testload):
        age_ids, input_ids, posi_ids, segment_ids, attMask, targets, _ = batch
        targets = torch.tensor(mlb.transform(targets.numpy()), dtype=torch.float32)
        
        age_ids = age_ids.to(global_params['device'])
        input_ids = input_ids.to(global_params['device'])
        posi_ids = posi_ids.to(global_params['device'])
        segment_ids = segment_ids.to(global_params['device'])
        attMask = attMask.to(global_params['device'])
        targets = targets.to(global_params['device'])
        
        with torch.no_grad():
            loss, logits = model(input_ids, age_ids, segment_ids, posi_ids,attention_mask=attMask, labels=targets)
        logits = logits.cpu()
        targets = targets.cpu()
        
        tr_loss += loss.item()
        y_label.append(targets)
        y.append(logits)

    y_label = torch.cat(y_label, dim=0)
    y = torch.cat(y, dim=0)

    aps, roc, output, label = precision_test(y, y_label)
    return aps, roc, tr_loss

In [ ]:
# ============ QUICK FINE-TUNING: 10 EPOCHS ============
print("\n" + "="*60)
print("STARTING QUICK FINE-TUNING - 10 EPOCHS")
print("="*60)
print("Expected time: 20-30 minutes")
print("="*60 + "\n")

# ADD THIS: Setup logging to file
import sys
log_file = os.path.join(global_params['output_dir'], 'nextvisit_training_log.txt')
print(f"Logging training results to: {log_file}")

class Logger:
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "w")

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()  # Ensure it's written immediately

    def flush(self):
        self.terminal.flush()
        self.log.flush()

# Redirect stdout
sys.stdout = Logger(log_file)

best_aps = 0.0
total_start = time.time()

for e in range(25):  # Only 10 epochs instead of 50
    print(f"\n{'='*60}")
    print(f"EPOCH {e+1}/10")
    print(f"{'='*60}")
    
    epoch_time = train(e)
    aps, roc, test_loss = evaluation()
    
    print(f"\nEpoch {e+1} Results:")
    print(f"  APS (Average Precision Score): {aps:.4f}")
    print(f"  ROC-AUC: {roc:.4f}")
    print(f"  Test Loss: {test_loss:.4f}")
    print(f"  Epoch Time: {epoch_time/60:.2f} minutes")
    
    if aps > best_aps:
        print(f"  ✓ New best APS! (previous: {best_aps:.4f})")
        print("  Saving model...")
        model_to_save = model.module if hasattr(model, 'module') else model
        output_model_file = os.path.join(global_params['output_dir'], global_params['best_name'])
        create_folder(global_params['output_dir'])
        torch.save(model_to_save.state_dict(), output_model_file)
        best_aps = aps
        print(f"  Model saved to: {output_model_file}")

total_time = (time.time() - total_start) / 60
print("\n" + "="*60)
print(f"QUICK FINE-TUNING COMPLETE!")
print("="*60)
print(f"Total time: {total_time:.2f} minutes")
print(f"Best APS achieved: {best_aps:.4f}")
print(f"Final model: {global_params['output_dir']}{global_params['best_name']}")
print("\nNote: For better results, use the full training notebooks.")
print("="*60)


STARTING QUICK FINE-TUNING - 10 EPOCHS
Expected time: 20-30 minutes

Logging training results to: ../data/models/quick/nextvisit_training_log.txt

EPOCH 1/10
epoch: 0	| Step: 0/899	| Loss: 0.6942	| APS: 0.0466	| Time: 9.1s
epoch: 0	| Step: 50/899	| Loss: 0.5259	| APS: 0.1568	| Time: 22.7s
epoch: 0	| Step: 100/899	| Loss: 0.3808	| APS: 0.2970	| Time: 37.6s
epoch: 0	| Step: 150/899	| Loss: 0.3064	| APS: 0.3103	| Time: 53.2s
epoch: 0	| Step: 200/899	| Loss: 0.2621	| APS: 0.3160	| Time: 69.0s
epoch: 0	| Step: 250/899	| Loss: 0.2327	| APS: 0.3300	| Time: 83.2s
epoch: 0	| Step: 300/899	| Loss: 0.2116	| APS: 0.3601	| Time: 97.4s
epoch: 0	| Step: 350/899	| Loss: 0.1956	| APS: 0.3447	| Time: 111.9s
epoch: 0	| Step: 400/899	| Loss: 0.1831	| APS: 0.3347	| Time: 125.3s
epoch: 0	| Step: 450/899	| Loss: 0.1731	| APS: 0.3524	| Time: 138.9s
epoch: 0	| Step: 500/899	| Loss: 0.1647	| APS: 0.3430	| Time: 155.5s
epoch: 0	| Step: 550/899	| Loss: 0.1577	| APS: 0.3316	| Time: 173.4s
epoch: 0	| Step: 600/899


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<string>", line 1, in <module>
  File "/opt/anaconda3/envs/my_env/lib/python3.9/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/anaconda3/envs/my_env/lib/python3.9/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
  File "/opt/anaconda3/envs/my_env/lib/python3.9/site-packages/torch/__init__.py", line 1382, in <module>
    from .functional import *  # noqa: F403
  File "/opt/anaconda3/envs/my_env/lib/python3.9/site-pa


Epoch 12 Results:
  APS (Average Precision Score): 0.3434
  ROC-AUC: 0.9032
  Test Loss: 8.2702
  Epoch Time: 5.85 minutes

EPOCH 13/10



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<string>", line 1, in <module>
  File "/opt/anaconda3/envs/my_env/lib/python3.9/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/anaconda3/envs/my_env/lib/python3.9/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
  File "/opt/anaconda3/envs/my_env/lib/python3.9/site-packages/torch/__init__.py", line 1382, in <module>
    from .functional import *  # noqa: F403
  File "/opt/anaconda3/envs/my_env/lib/python3.9/site-pa

epoch: 12	| Step: 0/899	| Loss: 0.0677	| APS: 0.3620	| Time: 9.6s
epoch: 12	| Step: 50/899	| Loss: 0.0730	| APS: 0.3625	| Time: 22.9s
epoch: 12	| Step: 100/899	| Loss: 0.0728	| APS: 0.3507	| Time: 37.0s
epoch: 12	| Step: 150/899	| Loss: 0.0727	| APS: 0.3466	| Time: 51.0s
epoch: 12	| Step: 200/899	| Loss: 0.0727	| APS: 0.3223	| Time: 64.4s
epoch: 12	| Step: 250/899	| Loss: 0.0727	| APS: 0.3479	| Time: 78.1s
epoch: 12	| Step: 300/899	| Loss: 0.0726	| APS: 0.3487	| Time: 91.8s
epoch: 12	| Step: 350/899	| Loss: 0.0726	| APS: 0.3282	| Time: 106.4s
epoch: 12	| Step: 400/899	| Loss: 0.0725	| APS: 0.3565	| Time: 120.5s
epoch: 12	| Step: 450/899	| Loss: 0.0725	| APS: 0.3577	| Time: 134.1s
epoch: 12	| Step: 500/899	| Loss: 0.0726	| APS: 0.3568	| Time: 148.7s
epoch: 12	| Step: 550/899	| Loss: 0.0725	| APS: 0.3608	| Time: 164.0s
epoch: 12	| Step: 600/899	| Loss: 0.0725	| APS: 0.3253	| Time: 179.1s
epoch: 12	| Step: 650/899	| Loss: 0.0726	| APS: 0.3444	| Time: 192.7s
epoch: 12	| Step: 700/899	| Los

...

epoch: 22	| Step: 0/899	| Loss: 0.0705	| APS: 0.4246	| Time: 11.2s
epoch: 22	| Step: 50/899	| Loss: 0.0683	| APS: 0.3927	| Time: 28.5s
epoch: 22	| Step: 100/899	| Loss: 0.0684	| APS: 0.3970	| Time: 45.9s
epoch: 22	| Step: 150/899	| Loss: 0.0685	| APS: 0.4047	| Time: 64.6s
epoch: 22	| Step: 200/899	| Loss: 0.0685	| APS: 0.3994	| Time: 86.6s
epoch: 22	| Step: 250/899	| Loss: 0.0684	| APS: 0.3982	| Time: 110.9s
epoch: 22	| Step: 300/899	| Loss: 0.0684	| APS: 0.4000	| Time: 134.8s
epoch: 22	| Step: 350/899	| Loss: 0.0684	| APS: 0.3903	| Time: 152.4s
epoch: 22	| Step: 400/899	| Loss: 0.0684	| APS: 0.3764	| Time: 168.6s
epoch: 22	| Step: 450/899	| Loss: 0.0684	| APS: 0.3996	| Time: 183.8s
epoch: 22	| Step: 500/899	| Loss: 0.0684	| APS: 0.4099	| Time: 202.4s
epoch: 22	| Step: 550/899	| Loss: 0.0684	| APS: 0.3961	| Time: 218.0s
epoch: 22	| Step: 600/899	| Loss: 0.0684	| APS: 0.4043	| Time: 233.1s
epoch: 22	| Step: 650/899	| Loss: 0.0684	| APS: 0.3955	| Time: 247.5s
epoch: 22	| Step: 700/899	| 


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<string>", line 1, in <module>
  File "/opt/anaconda3/envs/my_env/lib/python3.9/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/anaconda3/envs/my_env/lib/python3.9/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
  File "/opt/anaconda3/envs/my_env/lib/python3.9/site-packages/torch/__init__.py", line 1382, in <module>
    from .functional import *  # noqa: F403
  File "/opt/anaconda3/envs/my_env/lib/python3.9/site-pa


Epoch 23 Results:
  APS (Average Precision Score): 0.4054
  ROC-AUC: 0.9190
  Test Loss: 7.6980
  Epoch Time: 5.56 minutes
  ✓ New best APS! (previous: 0.4023)
  Saving model...
  Model saved to: ../data/models/quick/behrt_nextvisit_ccsr_quick.pt

EPOCH 24/10



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<string>", line 1, in <module>
  File "/opt/anaconda3/envs/my_env/lib/python3.9/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/anaconda3/envs/my_env/lib/python3.9/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
  File "/opt/anaconda3/envs/my_env/lib/python3.9/site-packages/torch/__init__.py", line 1382, in <module>
    from .functional import *  # noqa: F403
  File "/opt/anaconda3/envs/my_env/lib/python3.9/site-pa

epoch: 23	| Step: 0/899	| Loss: 0.0707	| APS: 0.4085	| Time: 1021.3s
